# Notebook 4: Variant Effect Prediction with Genomic Language Models

**BMI 511 — Spring 2026**
**Instructor:** Pratik Dutta, Ph.D. | Department of Biomedical Informatics, Stony Brook University

---

## Learning objectives

1. Define the **variant effect prediction (VEP)** problem for non-coding regulatory variants.
2. Score a single-nucleotide variant using a gLM's **probability shift** and **embedding shift**.
3. Compute **log-likelihood ratios (LLR)** with a masked-language-model head.
4. Rank a panel of SNPs by predicted functional impact.
5. Understand how this generalizes to the **DeepVRegulome** framework used in our lab for disease-variant prioritization.

> Runtime: **T4 GPU** recommended. This notebook uses only inference, no training.


## 1. The problem

Most disease-associated variants identified by GWAS lie in **non-coding** regions. They don't change a protein sequence; instead they may disrupt a transcription factor binding site, a splice signal, or a chromatin loop anchor. We need computational tools to predict which of the millions of candidate variants are **functionally impactful**.

A genomic language model offers two natural scoring schemes:

### 1.1 Embedding shift (semantic distance)
Embed the **reference** sequence and the **alternative** sequence (with the variant substituted). If the variant meaningfully changes the model's internal representation, the two embeddings will be far apart.

$$
\text{Shift}(ref, alt) = 1 - \cos\big(\mathbf{e}_{ref},\ \mathbf{e}_{alt}\big)
$$

### 1.2 Log-likelihood ratio (generative score)
Using the masked-LM head, compute how likely the model thinks each allele is at the variant position, given the surrounding context:

$$
\text{LLR}(ref \to alt) = \log P(\text{alt} \mid \text{context}) - \log P(\text{ref} \mid \text{context})
$$

A large-magnitude LLR means the model is confident that one allele fits the regulatory context better than the other — a signal of functional impact.

This is the **core mechanism** behind several recent tools (GPN, Nucleotide Transformer VEP, DeepVRegulome), just with different task heads and fine-tuning strategies.

## 2. Setup

In [ ]:
!pip install -q transformers==4.44.2 datasets einops biopython scikit-learn

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 3. Load two DNABERT-2 heads

In [ ]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Encoder -> for embedding-based scoring
from transformers import BertModel, BertConfig
config = BertConfig.from_pretrained(MODEL_NAME)
encoder = BertModel.from_pretrained(
    MODEL_NAME,
    config=config,
).to(device)
encoder.eval()

# MaskedLM head -> for likelihood-based scoring
from transformers import BertForMaskedLM
mlm = BertForMaskedLM.from_pretrained(
    MODEL_NAME,
    config=config,
).to(device)
mlm.eval()

print("Both model heads loaded.")

## 4. A panel of candidate variants

We construct a small panel where we know the *expected* impact:

| ID | Context | Variant | Expected impact |
| --- | --- | --- | --- |
| V1 | TATA box `TATAAAA` at promoter | T→G (destroys TATA) | **High** |
| V2 | TATA box `TATAAAA` | A→T (still within TATA tolerance) | Low–medium |
| V3 | Random intergenic | G→C | **Low** |
| V4 | GC-rich CpG island | C→T (disrupts CpG) | Medium |
| V5 | Middle of a long polyA tract | A→G | Low–medium |
| V6 | Splice donor `GT` dinucleotide | G→A (breaks splice site) | **High** |

We embed each reference/alternative pair and also score it with the MLM head.

In [ ]:
variants = [
    # id, reference_sequence, position (0-based), ref_allele, alt_allele, note
    ("V1_TATA_break",
     "GCGCGCGCGCGCGCGCGC" + "TATAAAA" + "GCGCGCGCGCGCGCGCGC",
     18, "T", "G",
     "TATA box → destroyed"),

    ("V2_TATA_mild",
     "GCGCGCGCGCGCGCGCGC" + "TATAAAA" + "GCGCGCGCGCGCGCGCGC",
     22, "A", "T",
     "TATA box → 1 bp change, likely tolerated"),

    ("V3_random",
     "ACGTACGTACGTACGTACGTACGTACGTACGTACGTACGT",
     15, "G", "C",
     "Random intergenic background"),

    ("V4_CpG",
     "CGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCG",
     20, "C", "T",
     "CpG island → CG destroyed"),

    ("V5_polyA",
     "GCGCGCGCGC" + "AAAAAAAAAAAAAAAAAAAA" + "GCGCGCGCGC",
     20, "A", "G",
     "Long polyA tract middle"),

    ("V6_splice_donor",
     "CAGTGACTGATCGATCGAT" + "GT" + "AAGCTAGCTAGCTAGCTAG",
     19, "G", "A",
     "5' splice donor GT → AT (breaks splicing)"),
]

df = pd.DataFrame(variants, columns=["id", "ref_seq", "pos", "ref", "alt", "note"])
df["alt_seq"] = df.apply(
    lambda r: r.ref_seq[:r.pos] + r.alt + r.ref_seq[r.pos + 1:], axis=1
)
# sanity check: reference base matches what we expect
assert (df.apply(lambda r: r.ref_seq[r.pos] == r.ref, axis=1)).all()
df

## 5. Scoring scheme 1 — embedding shift

In [ ]:
@torch.no_grad()
def embed(seq):
    enc = tok(seq, return_tensors="pt").to(device)
    out = encoder(**enc)
    h = out[0] if isinstance(out, tuple) else out.last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1).float()
    v = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
    return v.cpu().numpy().squeeze()

shifts = []
for _, r in df.iterrows():
    e_ref = embed(r.ref_seq)
    e_alt = embed(r.alt_seq)
    cos   = cosine_similarity(e_ref.reshape(1, -1), e_alt.reshape(1, -1))[0, 0]
    shifts.append(1.0 - cos)

df["embedding_shift"] = shifts
df[["id", "note", "embedding_shift"]].sort_values("embedding_shift", ascending=False)

## 6. Scoring scheme 2 — MLM log-likelihood ratio

We mask the variant position, ask the MLM head for its distribution over all tokens, and read off the log-probabilities of the `ref` and `alt` alleles.

> **Caveat.** DNABERT-2 uses BPE, so the variant position may not correspond to a single-nucleotide token. For a strictly rigorous LLR over single bases you would use a **character-level** gLM (e.g. GPN, Caduceus) or a k-mer model (original DNABERT with k=3). Here we illustrate the idea: we tokenize the full sequence, locate the token that covers the variant position, mask it, and compare the probabilities assigned to the ref-allele token vs alt-allele token. For teaching, this captures the right intuition.


In [ ]:
@torch.no_grad()
def llr_via_mlm(ref_seq, alt_seq, pos):
    """Approximate log-likelihood ratio using the BPE MLM head.

    Strategy:
      - Tokenize both full sequences.
      - Find the token in the reference that spans position `pos` via offset mapping.
      - Mask that token, run the MLM, and take log P(ref_token) and log P(alt_token_at_same_position).
    """
    enc_ref = tok(ref_seq, return_tensors="pt", return_offsets_mapping=True).to(device)
    offsets = enc_ref.pop("offset_mapping")[0].tolist()   # list of (start, end) pairs

    # locate token index covering `pos`
    tok_idx = None
    for i, (a, b) in enumerate(offsets):
        if a <= pos < b:
            tok_idx = i; break
    if tok_idx is None:
        return np.nan, "variant falls in a special token"

    ref_token_id = enc_ref["input_ids"][0, tok_idx].item()

    # corresponding alt token id: tokenize the alt sequence and read the same offset
    enc_alt = tok(alt_seq, return_tensors="pt", return_offsets_mapping=True).to(device)
    alt_offsets = enc_alt.pop("offset_mapping")[0].tolist()
    alt_tok_idx = None
    for i, (a, b) in enumerate(alt_offsets):
        if a <= pos < b:
            alt_tok_idx = i; break
    if alt_tok_idx is None:
        return np.nan, "alt token not found"
    alt_token_id = enc_alt["input_ids"][0, alt_tok_idx].item()

    # mask the variant token in the reference-context input and score both alleles
    masked = enc_ref["input_ids"].clone()
    masked[0, tok_idx] = tok.mask_token_id
    logits = mlm(input_ids=masked, attention_mask=enc_ref["attention_mask"]).logits   # (1, T, V)
    logp   = F.log_softmax(logits[0, tok_idx], dim=-1)                                # (V,)

    log_p_ref = logp[ref_token_id].item()
    log_p_alt = logp[alt_token_id].item()
    return log_p_alt - log_p_ref, f"tok_ref={tok.convert_ids_to_tokens(ref_token_id)}, tok_alt={tok.convert_ids_to_tokens(alt_token_id)}"

llrs, notes = [], []
for _, r in df.iterrows():
    llr, detail = llr_via_mlm(r.ref_seq, r.alt_seq, r.pos)
    llrs.append(llr); notes.append(detail)

df["llr"] = llrs
df["llr_abs"] = np.abs(llrs)
df["token_detail"] = notes
df[["id", "note", "embedding_shift", "llr", "llr_abs", "token_detail"]]

**Reading the scores.**
* `embedding_shift`: the larger, the more the variant disrupts the model's contextual representation.
* `llr`: negative means the model prefers the **reference** allele at this position; positive means it prefers the **alt** allele. `|llr|` is the magnitude of preference.
* A variant with both a large `embedding_shift` and a large `|llr|` is a strong candidate for functional impact.

## 7. Visual comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = df.sort_values("embedding_shift", ascending=True).index
axes[0].barh(df.loc[order, "id"], df.loc[order, "embedding_shift"], color="#4c72b0")
axes[0].set_xlabel("1 − cosine(ref, alt)")
axes[0].set_title("Embedding shift")

order2 = df.sort_values("llr_abs", ascending=True).index
axes[1].barh(df.loc[order2, "id"], df.loc[order2, "llr_abs"], color="#dd8452")
axes[1].set_xlabel("|log P(alt) − log P(ref)|")
axes[1].set_title("MLM log-likelihood ratio (absolute)")

for ax in axes:
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Do the two scores agree?

They measure related but non-identical things. Let's plot them against each other.


In [ ]:
plt.figure(figsize=(7, 6))
for _, r in df.iterrows():
    plt.scatter(r.embedding_shift, r.llr_abs, s=80, edgecolor="black")
    plt.annotate(r.id, (r.embedding_shift, r.llr_abs),
                 xytext=(5, 5), textcoords="offset points", fontsize=9)
plt.xlabel("Embedding shift  (1 − cos)")
plt.ylabel("|MLM log-likelihood ratio|")
plt.title("Two complementary variant-effect scores")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

from scipy.stats import spearmanr
rho, p = spearmanr(df.embedding_shift, df.llr_abs)
print(f"Spearman ρ between the two scores: {rho:.3f}   (p = {p:.3g})")

## 9. From this notebook to real variant prioritization

What we did here is the minimum viable VEP pipeline. A production-grade system such as **DeepVRegulome** extends it in three important ways:

1. **Task-specific heads**: instead of (or in addition to) the generic MLM head, fine-tune the encoder on specific regulatory tasks — transcription factor binding, histone marks, open chromatin, splice sites. Each task head gives a separate ref / alt score and a signed "regulatory switch" call.

2. **Hundreds of fine-tuned models**: DeepVRegulome uses **462 fine-tuned models** (458 transcription factors + 4 histone marks). A single variant is scored against every model, producing a multi-dimensional functional fingerprint.

3. **Variant-in-disease context**: scores are aggregated per gene and per disease cohort (e.g. GWAS variants in AD / PD / ASD), and prioritized by combining effect size with tissue-appropriate regulatory annotations.

The Streamlit app and Python package are openly available:
* GitHub: `DavuluriLab/DeepVRegulome`
* PyPI: `pip install deepvregulome`
* Demo: `deepvregulome.streamlit.app`

## 10. Exercises

1. Create a variant that **creates** a TATA box in a random background (`ACGT` → `ATATAAAAG` context). What happens to the two scores?
2. Run the LLR scoring using `zhihan1996/DNA_bert_6` (original DNABERT with **k=6 overlapping tokens**) instead of DNABERT-2. Does nucleotide-level interpretation become cleaner?
3. Download 10 ClinVar "Pathogenic" vs 10 "Benign" non-coding SNVs, score them with both methods, and check whether Pathogenic variants tend to have larger scores.

## 11. Recap

* gLMs score variants without any task-specific training via **embedding shift** and **MLM log-likelihood ratios**.
* The two scores are correlated but complementary — use both.
* Real-world VEP adds fine-tuned task heads, per-tissue regulatory annotations, and disease context (DeepVRegulome).

**This concludes the GLM module of BMI 511.** In four notebooks we have gone from "what is a token?" to scoring regulatory variants — the same recipe that powers contemporary genome-interpretation research.